# Partial / lazy reads from the APAC NWP Forecast Silver cube

Confirms the Silver format (Zarr Mode A + sharding) supports **loading only the chunks you
ask for** — e.g. only Taiwan, or only a few variables — instead of the whole multi-TB cube.

- Cube: `hf://datasets/jimtseng/apac-nwp-forecast/<model>_silver.zarr`
- Dims: `(run_init, lead, latitude, longitude)`; `valid_time = run_init + lead`
- Each variable is its own array; each run is its own shard → selections read only their bytes.

**Requirements:** `pip install "zarr>=3" xarray huggingface_hub numpy`

In [1]:
import time, warnings, numpy as np, xarray as xr
warnings.filterwarnings("ignore")

CUBE = "hf://datasets/jimtseng/apac-nwp-forecast/dwd_icon_silver.zarr"   # or jma_msm_silver.zarr

def timed(label, fn):
    t0 = time.perf_counter(); out = fn(); print(f"[{time.perf_counter()-t0:6.2f}s] {label}"); return out

## 1. Open lazily — nothing is downloaded yet
`open_zarr` only reads metadata. The arrays are dask-backed placeholders.

In [ ]:
ds = timed("open_zarr (metadata only)", lambda: xr.open_zarr(CUBE, consolidated=True))
print("dims:", dict(ds.sizes))
print("lazy (dask-backed)?", ds["temperature_2m_celsius"].chunks is not None)

full_gb = sum(v.size * v.dtype.itemsize for v in ds.data_vars.values()) / 1e9
print(f"full cube, all vars, uncompressed: ~{full_gb:,.0f} GB")

filled = np.where(ds["slot_filled"].values == 1)[0]   # runs that actually have data
print("filled runs:", len(filled), "/", ds.sizes["run_init"])

: 

## 2. Only Taiwan + 2 variables + a few runs
Select a Taiwan bounding box, two variables, five runs, first 24 lead hours — then `.load()`. Only those chunks are fetched (sub-MB), not the multi-TB cube.

In [3]:
tw = ds.sel(latitude=slice(21.8, 25.4), longitude=slice(119.9, 122.1))   # Taiwan bbox
print("Taiwan grid:", tw.sizes["latitude"], "x", tw.sizes["longitude"],
      "(full:", ds.sizes["latitude"], "x", ds.sizes["longitude"], ")")

sub = tw[["shortwave_radiation_wattPerSquareMetre", "temperature_2m_celsius"]] \
        .isel(run_init=filled[:5], lead=slice(0, 24))
loaded = timed("sub.load()  (reads only these chunks)", lambda: sub.load())

mb = sum(v.values.nbytes for v in loaded.data_vars.values()) / 1e6
print(f"loaded {mb:.2f} MB  <-- a Taiwan window, not {full_gb:,.0f} GB")
loaded

Taiwan grid: 29 x 17 (full: 721 x 497 )


[  6.58s] sub.load()  (reads only these chunks)
loaded 0.47 MB  <-- a Taiwan window, not 10,109 GB


<xarray.Dataset> Size: 478kB
Dimensions:                                 (run_init: 5, lead: 24,
                                             latitude: 29, longitude: 17)
Coordinates:
  * run_init                                (run_init) datetime64[ns] 40B 202...
  * lead                                    (lead) int32 96B 1 2 3 ... 22 23 24
  * latitude                                (latitude) float32 116B 21.88 ......
  * longitude                               (longitude) float32 68B 120.0 ......
    elevation                               (latitude, longitude) float32 2kB ...
    location_id                             (latitude, longitude) int32 2kB 2...
Data variables:
    shortwave_radiation_wattPerSquareMetre  (run_init, lead, latitude, longitude) float32 237kB ...
    temperature_2m_celsius                  (run_init, lead, latitude, longitude) float32 237kB ...
Attributes:
    title:       Combined NWP forecast cube (per-run, region-written)
    model:       dwd_icon
    grid:        721x497
    layout:      (run_init, lead, latitude, longitude); valid_time = run_init...
    lead_units:  hours since run_init

## 3. Only one variable
Each weather variable is a separate array (its own shards), so selecting `temperature_2m_celsius` never touches `cloud_cover`'s bytes.

In [4]:
pt = timed("Taipei point, all 180 lead, 1 run",
            lambda: ds["temperature_2m_celsius"]
                      .sel(latitude=25.0, longitude=121.5, method="nearest")
                      .isel(run_init=filled[10]).load())
print("shape:", pt.shape, "| finite:", int(np.isfinite(pt.values).sum()), "/", pt.size)
print("temperature (degC):", [round(float(x), 1) for x in pt.values[:8]], "...")

[  1.26s] Taipei point, all 180 lead, 1 run
shape: (180,) | finite: 180 / 180
temperature (degC): [17.6, 17.6, 17.6, 17.5, 17.5, 17.3, 17.4, 17.5] ...


## 4. Chunk-scoped cost: point vs region
Both are cheap because each touches only a few chunks — the whole point of Mode A + sharding.

In [5]:
v = "temperature_2m_celsius"; j = int(filled[10])
timed("point series (1 lat, 1 lon, all lead, 1 run)",
      lambda: ds[v].sel(latitude=25, longitude=121.5, method="nearest").isel(run_init=j).load())
timed("Taiwan region (all lead, 1 run)",
      lambda: ds[v].sel(latitude=slice(21.8, 25.4), longitude=slice(119.9, 122.1)).isel(run_init=j).load())
print("\nPartial/lazy reads confirmed: region + variable + run selections each read only their chunks.")

[  2.67s] point series (1 lat, 1 lon, all lead, 1 run)


[  2.10s] Taiwan region (all lead, 1 run)

Partial/lazy reads confirmed: region + variable + run selections each read only their chunks.


## Notes
- `hf://` streams shards on demand over the network. For heavy/repeated use, download the cube
  (or specific variables) locally first, then open the local path.
- Use `ds.isel(run_init=(ds["slot_filled"]==1))` to skip empty future slots.
- "All forecasts valid at time *T*" is the anti-diagonal `run_init + lead == T`.
- Swap `CUBE` to `jma_msm_silver.zarr` for the 5 km Japan-domain model (13 vars, no snow).